# CLIFFGUARD on Google Colab

**TL;DR.**
- Runs real Fold A calibration and Fold B cliff measurement on a Colab GPU — no local hardware required.
- Free T4 (16 GB) handles 1B–3B in FP16/NF4; Pro A100 (40 GB) handles up to 8B FP16 and 70B GGUF.
- Every `(model, scheme)` pair checkpoints to Google Drive after it finishes; a disconnected session loses at most one scheme of work.

**What you'll get.**
1. Refusal direction `r̂` and per-scheme threshold `τ_q` in `fold_a/` (Arditi diff-in-means calibration).
2. Cliff metrics — `Δ_cliff`, `Δ_W-cliff`, `Δ_B-cliff` — in `fold_b/`, plus an `H1` accept/reject verdict.
3. A persistent run directory at `/content/drive/MyDrive/cliffguard/results/<run_id>/`.

> **Sessions disconnect.** Free Colab can vanish at any time. This notebook re-runs safely; the checkpoint mechanism skips any `(model, scheme)` that finished before the disconnect.

> **Read this before running.** Several past failure modes are now guarded:
> - `verify_fold_a_complete` raises if you try to run Fold B with an incomplete Fold A — no more silent `cliff(FP16, FP16) = 0` results.
> - `ensure_repo_cwd()` is called at the top of every cell that uses relative paths; `%cd` drift after a kernel restart no longer breaks scripts.
> - `torch_cleanup()` runs between schemes so NF4 doesn't fail with an opaque "weight conversion" error after FP16.
> - The behavioural metric `Δ_B-cliff` in Phase B is a **PROBE-RM margin-threshold proxy**, not StrongREJECT + Llama-Guard-3-8B (paper §11.3 requirement is Phase C).

Full setup walkthrough lives in [`docs/setup_colab.md`](../docs/setup_colab.md).

In [ ]:
!nvidia-smi
import torch
print(f"CUDA: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
free, total = torch.cuda.mem_get_info() if torch.cuda.is_available() else (0, 0)
print(f"VRAM free: {free / 1024**3:.2f} GB / {total / 1024**3:.2f} GB")

## What you should see in C1

| Hardware | Typical free VRAM | What it can run |
|---|---|---|
| T4 (free) | ~14–15 GB of 16 GB | Llama-3.2-1B/3B FP16, 8B NF4 |
| L4 (Pro) | ~22–23 GB of 24 GB | Llama-3.1-8B FP16, 13B NF4 |
| A100 40 GB (Pro) | ~38 GB free | 13B FP16, 70B NF4 |
| A100 80 GB (Pro+) | ~78 GB free | 30B FP16, 70B full |

If VRAM shows 0 GB you're on CPU — switch via **Runtime → Change runtime type → T4 GPU** and re-run C1.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/cliffguard/results
!mkdir -p /content/drive/MyDrive/cliffguard/models
!mkdir -p /content/drive/MyDrive/cliffguard/datasets/folds/fold_a
!mkdir -p /content/drive/MyDrive/cliffguard/datasets/folds/fold_b

## Drive layout

`/content/drive/MyDrive/cliffguard/` is the persistent staging area. The notebook treats it as the source of truth — anything written under `/content/CLIFFGUARD/artifacts/` is copied here after every scheme.

```
/content/drive/MyDrive/cliffguard/
├── datasets/folds/fold_a/   ← linked into <repo>/data/folds/fold_a/ in cell C16
├── datasets/folds/fold_b/   ← linked into <repo>/data/folds/fold_b/ in cell C16
├── models/                  ← (optional) HF cache redirect
└── results/<run_id>/
    ├── run_metadata.json
    ├── fold_a/   (calibration_summary.json, r_hat_*.npz, checkpoint.json)
    └── fold_b/   (cliff_results.json, checkpoint.json)
```

In [ ]:
%cd /content
# EDIT: change <owner> to your GitHub user/org (e.g. parnish007) BEFORE running.
REPO_OWNER = '<owner>'   # <-- EDIT THIS
!git clone https://github.com/{REPO_OWNER}/CLIFFGUARD.git || (cd CLIFFGUARD && git pull)
%cd /content/CLIFFGUARD
!git log -1 --oneline

> **Edit `REPO_OWNER` above** before running. If the placeholder `<owner>` is left in, the `git clone` will fail.

In [ ]:
# Install directly into Colab's system Python — `uv venv` activation inside
# `!` cells is inconsistent on Colab and takes ~10 min. Plain pip is ~3 min.
#
# numpy<2 + matched bitsandbytes is REQUIRED to avoid the NF4 "weight
# conversion" error that broke previous runs on Llama-3.2 models.
%cd /content/CLIFFGUARD
!pip install -q "numpy<2"
!pip install -q -e .
!pip install -q "torch" "transformers>=4.45,<4.50" "bitsandbytes>=0.43,<0.45" accelerate sentencepiece protobuf datasets
print("--- install done ---")

In [ ]:
# Pre-built CUDA wheel — installs in ~5s vs ~5–10 min for the CMAKE source build.
# Matches Colab's CUDA 12.x runtime; falls back to the source build only if
# the wheel is unavailable for your CUDA version.
!pip install -q llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122 \
  || CMAKE_ARGS="-DLLAMA_CUDA=on" pip install --force-reinstall --no-cache-dir llama-cpp-python -q

## HuggingFace authentication

Meta's Llama models are gated. Two steps, once per HF account:

1. **Accept each license** (click *Agree and access repository* on each page):
    - `meta-llama/Llama-3.2-1B-Instruct`
    - `meta-llama/Llama-3.2-3B-Instruct`
    - `meta-llama/Llama-Guard-3-8B`
    - `mistralai/Mistral-7B-Instruct-v0.3`
2. **Generate a read token** at [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens).

**Preferred — add the token as a Colab Secret.** Click the key icon (🔑) in the left sidebar → *Add new secret* → **Name: `HF_TOKEN`**, paste your token, toggle *Notebook access* on. Cell C9 below picks it up automatically and falls back to the interactive prompt only if the secret is absent.

In [ ]:
from huggingface_hub import login
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
    login(token=token, add_to_git_credential=False)
    print('[hf_login] using Colab Secret HF_TOKEN')
except Exception as exc:
    print(f'[hf_login] no Colab Secret found ({type(exc).__name__}); falling back to interactive login')
    login()  # interactive — paste token below

In [ ]:
import sys
sys.path.insert(0, '/content/CLIFFGUARD/notebooks')
import colab_helper as ch
ch.ensure_repo_cwd()   # <-- CWD will always be /content/CLIFFGUARD after this
ch.banner()

## Smoke test (no GPU, no datasets)

`scripts/dry_run.py` exercises the full pipeline shape using deterministic synthetic arrays. Under one second, every gate produces a verdict, the CONDUCTOR aggregates to a single BLOCK/ALLOW decision. If this fails, stop here and check the install.

In [ ]:
ch.ensure_repo_cwd()
!python scripts/dry_run.py --tier A --scheme FP16
!python scripts/dry_run.py --tier C --scheme GGUF_Q3_K_M

## Datasets — link to Drive, then download

First-run download is ~5–10 min. Subsequent sessions skip the download entirely because `data/` is a symlink into Drive.

**Fold A** = Anthropic-HH-RLHF + OpenAssistant OASST1 (benign + refused turns). 500 of each is enough for an exploratory run; the paper minimum is 2000 (see `--max` flag).  
**Fold B** = walledai/AdvBench + JailbreakBench/JBB-Behaviors. Pulled by `ch.assemble_fold_b()` in cell C22.

In [ ]:
ch.ensure_repo_cwd()
ch.symlink_datasets_from_drive()
# Explicit --target-dir guarantees the JSONL lands inside the symlinked
# Drive folder regardless of CWD quirks during the script run.
!python scripts/download_fold_a.py --download --max 500 \
  --target-dir data/folds/fold_a
!ls -la data/folds/fold_a/

## Configure the run

`ch.choose_model()` reads free VRAM and picks a model + scheme set that fits (rule of thumb: ~2 GB per 1B params in FP16, ~0.5 GB per 1B in NF4):

| Free VRAM | Model | Schemes |
|---|---|---|
| ≥ 35 GB | `meta-llama/Llama-3.1-8B-Instruct` | `[FP16, NF4, AWQ_INT4]` |
| ≥ 14 GB | `meta-llama/Llama-3.2-3B-Instruct` | `[FP16, NF4]` |
| ≥ 8 GB | `meta-llama/Llama-3.2-1B-Instruct` | `[FP16, NF4]` |
| < 8 GB | `Qwen/Qwen2.5-0.5B-Instruct` | `[FP16]` (degraded) |

Override by editing the dict before running Fold A. **Fold B requires at least 2 schemes** — the cliff is a scheme-vs-scheme measurement.

In [ ]:
ch.ensure_repo_cwd()
config = ch.choose_model()
# You can override here. Example for forcing 1B + 3 schemes:
# config = {'model_id': 'meta-llama/Llama-3.2-1B-Instruct', 'layer': 8,
#           'schemes': ['FP16', 'NF4'], 'est_runtime_min': 15, 'vram_gb': config['vram_gb']}
print(config)

## Run Fold A — calibration with checkpointing

`run_fold_a_with_checkpoint` performs Arditi diff-in-means calibration for each scheme in `config['schemes']`. After every scheme it:
1. Writes `r̂` to `artifacts/runs/<run_id>/fold_a/r_hat_<model>_<scheme>.npz`.
2. Updates `fold_a/checkpoint.json` with the scheme as completed.
3. Calls `torch_cleanup()` so the next NF4 load isn't blocked by FP16 residue.
4. Syncs the new files to Drive.

Re-running the cell after a disconnect picks up the same `run_id` (matched by `model_id`) and skips any scheme already in `completed_schemes`. If a scheme fails mid-load, the checkpoint is **not** advanced and you can simply re-run.

In [ ]:
ch.ensure_repo_cwd()
ch.run_fold_a_with_checkpoint(config)

## Force a Drive sync

The helper already syncs after each scheme. Use this cell only if you've edited artifacts manually or want one more sync before disconnecting.

In [ ]:
ch.sync_artifacts_to_drive()

## Assemble the Fold B corpus

Downloads `walledai/AdvBench` + `JailbreakBench/JBB-Behaviors` into `data/folds/fold_b/`. Idempotent — skips if files already exist. The JSONL shape matches `cliffguard.eval.folds._load_jsonl_fold`.

> **Note on Fold B scope.** This corpus is a subset of paper §12.6 — JBB and AdvBench are present, HarmBench and ArtPrompt are not. Adequate for an exploratory Phase B run; a full paper-spec replication needs additional sources.

In [ ]:
ch.ensure_repo_cwd()
ch.assemble_fold_b()

## Fold-isolation audit

Asserts that no prompt appears in both Fold A and Fold B (SHA-256 disjointness). The paper requires this before any unblinding — if it fires, the Fold B result is contaminated and must be regenerated.

In [ ]:
ch.ensure_repo_cwd()
ch.fold_isolation_audit()

## Verify Fold A is complete before Fold B

Cliff measurement is degenerate if you have only one scheme (the cliff of a scheme against itself is identically zero). This guard fails loudly rather than letting Fold B silently return `cliff_boundary: null`.

In [ ]:
ch.ensure_repo_cwd()
ch.verify_fold_a_complete(config)

## Run Fold B — cliff measurement

Fold B loads the saved `r_hat_*.npz` from `fold_a/` (no re-calibration), then for each scheme computes:
- `Δ_cliff(q, FP16)` — geometric distance between refusal directions.
- `Δ_W-cliff(q, FP16)` — Wasserstein distance between margin distributions.
- `Δ_B-cliff(q, FP16)` — change in attack-success rate above/below the calibrated τ_q threshold.

Result: `fold_b/cliff_results.json` with all three metrics and the `cliff_boundary` (first scheme where all metrics exceed κ = 0.25 — the H1 verdict for this model family).

> **Caveat on `Δ_B-cliff` (paper §11.3).** The behavioural metric here uses the **PROBE-RM margin threshold** as a behavioural surrogate (see `cliffguard/eval/five_fold_live.py:243-250`). The paper requires a real **StrongREJECT + Llama-Guard-3-8B joint judge**, which is tracked as Phase C work and is not active in this notebook.

In [ ]:
ch.ensure_repo_cwd()
ch.run_fold_b_with_checkpoint(config)
ch.sync_artifacts_to_drive()

## Inspect the run

Print the saved JSON to confirm Fold A τ_q thresholds and Fold B cliff metrics before downloading.

In [ ]:
ch.ensure_repo_cwd()
import json, pathlib
from cliffguard.eval.results_writer import list_runs
for run in list_runs(pathlib.Path('artifacts')):
    print('=', run.name)
    for name in ['fold_a/calibration_summary.json', 'fold_a/checkpoint.json',
                 'fold_b/cliff_results.json', 'fold_b/checkpoint.json']:
        p = run / name
        if p.exists():
            print(f'-- {name} --')
            print(json.dumps(json.load(open(p)), indent=2))

## Wrap-up

**Where results live.** `/content/drive/MyDrive/cliffguard/results/<run_id>/` survives disconnects. To download locally:
- Files panel → that path → right-click → *Download*, **or**
- `!zip -r /content/run.zip /content/drive/MyDrive/cliffguard/results/<run_id> && cp /content/run.zip /content/drive/MyDrive/`

**Resuming a killed session.**
1. Reconnect to a runtime.
2. Re-run cells **C1 through C10** in order (Drive mount, clone, install, HF login, helper import).
3. Re-run cell **C18** — `run_fold_a_with_checkpoint` reads the checkpoint in Drive and skips schemes already marked completed.
4. Re-run the Fold B cell. You lose **at most one scheme** of compute.

**Known limitations (not bugs).**
- Phase B `Δ_B-cliff` is a margin proxy, not StrongREJECT + Llama-Guard. Real judges land in Phase C.
- Fold B corpus here is JBB + AdvBench only (no HarmBench, no ArtPrompt).
- AWQ scheme only runs on Linux; the helper will skip it automatically on T4 if you add it to `config['schemes']`.

**Next.** See [`docs/setup_colab.md`](../docs/setup_colab.md) for troubleshooting and [`docs/cliffguard_complete_guide.md`](../docs/cliffguard_complete_guide.md) for the conceptual reference.